In [3]:
import torch
from torch import nn
from d2l import torch as d2l

In [3]:
def corr2d(X,K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0]-h+1, X.shape[1]-w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = (X[i:i+h, j:j +w]*K).sum() # elementwise mul
    return Y            

In [4]:
class Conv2d(nn.Module): # conv layer
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self,x):
        return corr2d(x,self.weight) + self.bias    

In [5]:
# create an image
X =  torch.ones((6,8))
X[:,2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [6]:
K = torch.tensor([[1.0,-1.0]])
Y=corr2d    (X,K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [7]:
corr2d(X.t(),K) # transposed image doesnt work here since it detects only vertical edges

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [8]:
conv2d = nn.LazyConv2d(1,kernel_size=(1,2), bias = False)
X =X.reshape((1,1,6,8))
Y =Y.reshape((1,1,6,7))
lr = 3e-2

for i in range(10):
    Y_hat =conv2d(X)
    l = (Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()

    conv2d.weight.data[:] -=lr*conv2d.weight.grad
    if(i+1)%2==0:
        print(f"epoch {i+1}, loss {l.sum():.3f}")


epoch 2, loss 10.555
epoch 4, loss 1.989
epoch 6, loss 0.423
epoch 8, loss 0.108
epoch 10, loss 0.033


In [ ]:
def comp_conv2d(conv2d, X):
    X = X.reshape((1,1)+X.shape)
    Y=conv2d(X) # requires 4d so we add 1,1 to 8,8
    return Y.reshape(Y.shape[2:]) # remove extra added dims

conv2d = nn.LazyConv2d(1,kernel_size=3,padding = 1)
X=torch.rand(size =(8,8))
comp_conv2d(conv2d, X)



tensor([[-5.1589e-02, -8.7344e-03,  1.0188e-01, -3.0978e-02,  3.0749e-02,
          7.6222e-02, -8.1373e-03, -1.5550e-01],
        [ 3.5298e-01,  3.2253e-01,  3.8993e-01,  2.4987e-01,  3.4382e-01,
          2.7539e-01,  4.5961e-01,  5.3241e-02],
        [ 3.8821e-01,  1.8295e-01,  2.3385e-01,  2.6902e-01,  1.3525e-01,
          1.7138e-01,  2.1208e-01,  4.0337e-01],
        [ 7.0441e-02,  2.1484e-01,  3.8624e-01,  3.1439e-01,  1.6589e-01,
          4.0467e-01,  1.4914e-01,  6.6238e-02],
        [ 9.3426e-02,  2.1314e-01,  2.3057e-01,  5.1038e-01,  2.7685e-01,
          4.3042e-02,  2.0233e-01,  1.1266e-01],
        [ 3.3511e-01,  2.3978e-01,  1.8981e-01,  2.5658e-01,  6.2283e-01,
          2.4073e-01,  2.1575e-01,  1.5904e-01],
        [ 3.6820e-01,  2.6940e-01,  2.2097e-01,  1.3362e-01,  4.4032e-01,
          4.8333e-01,  3.2957e-01,  4.4853e-02],
        [ 1.0746e-01,  2.0707e-01,  9.9172e-02,  4.3820e-04, -8.1617e-02,
         -1.5614e-02,  2.4907e-01,  2.2376e-01]], grad_fn=<ViewBa

In [1]:
def corr2d_multi_in(X,K):
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K))

In [4]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)


tensor([[ 56.,  72.],
        [104., 120.]])

In [5]:
def corr2d_multi_in_out(X,K):
    return torch.stack([corr2d_multi_in(X,k) for k in K], 0)